<a href="https://colab.research.google.com/github/Rakshitha004/DSP/blob/main/6thone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok bcrypt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 84.0 MB/s eta 0:00:00


In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 3s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [ ]:
%%writefile app.py
import streamlit as st
import hashlib, bcrypt, itertools, random, math, time, io
import pandas as pd

# ---------------- Utilities ----------------
def hash_str(s: str, algo: str) -> str:
    b = s.encode("utf-8")
    if algo == "MD5": return hashlib.md5(b).hexdigest()
    if algo == "SHA1": return hashlib.sha1(b).hexdigest()
    if algo == "SHA256": return hashlib.sha256(b).hexdigest()
    raise ValueError("Unsupported algo")

def verify_hash(candidate: str, target: str, algo: str) -> bool:
    if algo == "Bcrypt ($2b$)":
        try: return bcrypt.checkpw(candidate.encode(), target.encode())
        except: return False
    return hash_str(candidate, algo) == target

def password_strength(pw: str):
    lowers, uppers = any(c.islower() for c in pw), any(c.isupper() for c in pw)
    digits, symbols = any(c.isdigit() for c in pw), any(not c.isalnum() for c in pw)
    variety = sum([lowers, uppers, digits, symbols])
    charset_size = (26 if lowers else 0)+(26 if uppers else 0)+(10 if digits else 0)+(33 if symbols else 0)
    entropy = 0 if charset_size == 0 else round(len(pw)*math.log2(charset_size),1)
    if len(pw)<8 or variety<=1: return "Weak", entropy
    if len(pw)>=12 and variety>=3: return "Strong", entropy
    return "Medium", entropy

# ---------------- Streamlit UI ----------------
st.set_page_config(page_title="Password Attack Lab", page_icon="🔐")
st.title("🔐 Password Attack Lab (Colab GUI Demo)")

mode = st.radio("Target type", ["HASH (MD5/SHA1/SHA256/Bcrypt)", "PLAINTEXT"], horizontal=True)

algo = "SHA256"
target_hash, target_plain = "", ""
if mode.startswith("HASH"):
    algo = st.selectbox("Hash Algorithm", ["MD5","SHA1","SHA256","Bcrypt ($2b$)"])
    target_hash = st.text_input("Enter Target Hash")
else:
    target_plain = st.text_input("Enter Target Plaintext")

dictionary = st.text_area("Dictionary words (one per line)",
                          value="password\n123456\nP@ssw0rd\nadmin\nwelcome").splitlines()

if st.button("🚀 Run Dictionary Attack"):
    results=[]
    found=None
    for word in dictionary:
        if mode.startswith("HASH"):
            ok = verify_hash(word, target_hash, algo)
        else:
            ok = (word==target_plain)
        cat, ent = password_strength(word)
        results.append({"candidate":word,"match":ok,"category":cat,"entropy":ent})
        if ok: found=word
    df=pd.DataFrame(results)
    if found: st.success(f"✅ Found match: {found}")
    else: st.error("❌ No match found")
    st.dataframe(df)
    st.download_button("⬇️ Download CSV", data=df.to_csv(index=False).encode(),
                       file_name="analysis.csv", mime="text/csv")


Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙

⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://olive-tigers-argue.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.83.244.196:8501

